# World University Rankings 2026 – ETL desde Times Higher Education

Este notebook extrae, transforma y guarda los datos de **tres rankings publicados por _Times Higher Education_ (THE)** directamente desde su sitio web oficial:

1. **World University Rankings 2026** (ranking global).
2. **World University Rankings by Subject 2026 – Business & Economics** (ranking por disciplina).
3. **Latin America University Rankings 2026** (ranking regional para Latinoamérica).

El enfoque se mantiene fiel a la idea original (_web scraping_ con Selenium + BeautifulSoup) pero se adapta a la nueva estructura del sitio de THE (2026), donde las clases CSS son **hashes generados por CSS-in-JS** (`css-17iq9vf`, `css-1gidy1s`, `css-qpf2da`) que cambian en cada deploy. Por eso el parser trabaja **por posición de columna** dentro de la tabla.

> Origen de las URLs: archivo **`Web Scrapping from Times higher Education_Latest.txt`**.

## 1. Objetivo

Automatizar la captura de los rankings publicados por THE y dejar los datos listos para análisis posterior (comparación, visualización, modelado). Se usa _web scraping_ porque THE no expone un API público estable.

## 2. Prerrequisitos

* Python 3.9 o superior.
* Jupyter Notebook o cualquier entorno que permita ejecutar Python.
* Librerías: `beautifulsoup4`, `selenium`, `webdriver-manager`, `requests`, `pandas`, `lxml`.
* Navegador **Google Chrome** o **Chromium** instalado. `webdriver-manager` descarga el driver automáticamente (ya **no** hace falta un `chromedriver.exe` manual).

Instalar dependencias:
```bash
pip install -r requirements.txt
```

## 3. ¿Qué datos vamos a extraer?

La página muestra una tabla con ocho columnas fijas: **Rank, Name (nombre + país), Overall, Teaching, Research Environment, Research Quality, Industry, International Outlook**. La vista del usuario se ve así:

![Vista del ranking THE 2026](img/basic_page_01.PNG)
<br/><br/>


Abriendo DevTools (`F12`) vemos que cada fila es un `<tr class="group css-qpf2da">` con varios `<td class="css-17iq9vf">`; dentro de cada `<td>` de puntaje hay un `<div class="css-1gidy1s">` con el valor. Estos nombres de clase son **hashes** y cambian entre deploys, así que no nos apoyamos en ellos.

![Inspección del HTML en DevTools](img/page_code_02.PNG)
<br/><br/>

**Estrategia del parser**: obtener todos los `<td>` hijos directos de cada `<tr>` y leer por posición:

| Posición | Campo |
|---------|-------|
| 0 | Rank |
| 1 | Name + Country (el nombre es el primer texto visible, el país viene como segunda línea o en un enlace `/location/…`) |
| 2 | Overall |
| 3 | Teaching |
| 4 | Research Environment |
| 5 | Research Quality |
| 6 | Industry |
| 7 | International Outlook |

## 4. Implementación en Python

El sitio de THE carga la tabla mediante **AJAX/JavaScript**, por lo que una petición directa con `requests` devuelve la plantilla vacía. Usamos **Selenium** para dejar que el JS se ejecute y luego **BeautifulSoup** para el parsing.

In [ ]:
# Librerías estándar
import json
import re
import time
from urllib.parse import urljoin

# Librerías de terceros
import pandas as pd
import requests
from bs4 import BeautifulSoup as soup

# Selenium + webdriver-manager
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

### 4.1 Definir las URLs de los tres rankings

Mapeo de cada ranking al link, a su metodología y al nombre del archivo de salida.

In [ ]:
RANKINGS = {
    'world_2026': {
        'nombre': 'World University Rankings 2026',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/latest/world-ranking',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/methodology',
        'archivo_salida': 'the_world_ranking_2026.csv',
    },
    'business_economics_2026': {
        'nombre': 'Subject Ranking 2026 – Business & Economics',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/2026/subject-ranking/business-and-economics',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/world-university-rankings-subject-2026-methodology',
        'archivo_salida': 'the_subject_business_economics_2026.csv',
    },
    'latam_2026': {
        'nombre': 'Latin America University Rankings 2026',
        'url':    'https://www.timeshighereducation.com/world-university-rankings/2026/latin-america-university-rankings',
        'metodologia': 'https://www.timeshighereducation.com/world-university-rankings/latin-america-university-rankings-2026-methodology',
        'archivo_salida': 'the_latam_ranking_2026.csv',
    },
}

for k, v in RANKINGS.items():
    print(f"{k:25s} -> {v['nombre']}")

### 4.2 Inicializar el driver de Chrome

`webdriver-manager` descarga automáticamente la versión de `chromedriver` compatible con el Chrome instalado. Usamos modo `headless` para un scraping más liviano.

In [ ]:
def crear_driver(headless: bool = True) -> webdriver.Chrome:
    """Crea un driver de Chrome con las opciones recomendadas para scraping."""
    options = Options()
    if headless:
        options.add_argument('--headless=new')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument(
        'user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    )
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

### 4.3 Parser posicional de la tabla

La función `parsear_filas` recibe el HTML ya renderizado y devuelve la lista de diccionarios con los campos estandarizados. Detecta el país buscando enlaces a `/location/…` o el segundo bloque de texto de la celda de nombre; si no lo encuentra, queda vacío (se verá como `None` en el DataFrame).

In [ ]:
NUMERIC_RE = re.compile(r'^-?\d+(?:[.,]\d+)?$')


def _texto_limpio(nodo) -> str:
    return re.sub(r'\s+', ' ', nodo.get_text(' ', strip=True)).strip()


def _extraer_puntaje(td) -> str:
    """Devuelve el texto numérico de una celda de puntaje.
    Busca primero el <div> interno (patrón `css-1gidy1s`), cae a get_text."""
    if td is None:
        return None
    div = td.find('div')
    txt = _texto_limpio(div) if div else _texto_limpio(td)
    # Nos quedamos con la primera palabra si hay más de una (p. ej. "58.7 Data unavailable")
    if txt:
        primera = txt.split()[0]
        if NUMERIC_RE.match(primera.replace(',', '.')):
            return primera
    return txt or None


def _extraer_nombre_y_pais(td):
    """De la celda de nombre obtiene (nombre_universidad, pais, url_perfil).
    Reglas:
      - El nombre es el primer <a> no `/location/...` o el primer texto relevante.
      - El país viene en un <a href="/location/..."> o en un span/div secundario.
      - La URL de perfil es el primer <a> que no apunte a /location/.
    """
    nombre, pais, url_perfil = None, None, None
    if td is None:
        return nombre, pais, url_perfil
    for a in td.find_all('a'):
        href = a.get('href', '') or ''
        if '/location/' in href:
            if not pais:
                pais = _texto_limpio(a)
        else:
            if not nombre:
                nombre = _texto_limpio(a)
                url_perfil = href
    if not nombre:
        nombre = _texto_limpio(td)
    if not pais:
        # Intentamos tomar el segundo bloque de texto de la celda
        bloques = [b for b in td.stripped_strings]
        if len(bloques) >= 2 and bloques[0] == (nombre or ''):
            pais = bloques[1]
        elif len(bloques) >= 2:
            # a veces el nombre viene con saltos internos
            pais = bloques[-1] if bloques[-1] != nombre else None
    return nombre, pais, url_perfil


COLUMNAS_PUNTAJE = [
    'overall_score', 'teaching_score', 'research_env_score',
    'research_quality_score', 'industry_score', 'international_score',
]


def parsear_filas(html: str, base_url: str = 'https://www.timeshighereducation.com'):
    """Parser posicional: td[0]=rank, td[1]=name+country, td[2..7]=puntajes."""
    pagina = soup(html, 'html.parser')
    registros = []
    for tr in pagina.select('table tbody tr'):
        tds = tr.find_all('td', recursive=False)
        if len(tds) < 3:
            continue
        rank_txt = _texto_limpio(tds[0])
        if not rank_txt:
            continue
        nombre, pais, href = _extraer_nombre_y_pais(tds[1])
        registro = {
            'rank': rank_txt,
            'name': nombre,
            'country': pais,
            'url': urljoin(base_url, href) if href else None,
        }
        # Puntajes: 6 columnas desde td[2] hasta td[7] (puede haber menos si el ranking es subject/latam)
        for idx, columna in enumerate(COLUMNAS_PUNTAJE, start=2):
            td = tds[idx] if idx < len(tds) else None
            registro[columna] = _extraer_puntaje(td)
        registros.append(registro)
    return registros

### 4.4 Función principal de scraping

Carga la URL, espera a que aparezcan filas en la tabla, desplaza la página para forzar la hidratación completa de React y finalmente llama al parser posicional.

In [ ]:
def scrapear_ranking(driver: webdriver.Chrome, url: str, espera_seg: int = 30) -> pd.DataFrame:
    url_completa = url
    if '#' not in url_completa:
        url_completa = url + '#!/length/-1/sort_by/rank/sort_order/asc'
    print(f'→ Cargando {url_completa}')
    driver.get(url_completa)

    try:
        WebDriverWait(driver, espera_seg).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'table tbody tr'))
        )
    except Exception as e:
        print('  · Advertencia: no se detectó <tr> visible, seguimos ->', e)

    # Scroll para que la lista virtualizada de React termine de pintar todas las filas
    altura_prev = 0
    for _ in range(6):
        driver.execute_script('window.scrollTo(0, document.body.scrollHeight);')
        time.sleep(2)
        altura = driver.execute_script('return document.body.scrollHeight')
        if altura == altura_prev:
            break
        altura_prev = altura
    driver.execute_script('window.scrollTo(0, 0);')
    time.sleep(1)

    registros = parsear_filas(driver.page_source)
    print(f'  · Filas parseadas: {len(registros)}')
    df = pd.DataFrame(registros)
    if not df.empty:
        df['rank'] = df['rank'].astype(str)
    return df

### 4.5 Ejecutar el scraping de los tres rankings

Reutilizamos una sola sesión de Chrome para los tres rankings.

In [ ]:
driver = crear_driver(headless=True)
resultados = {}
try:
    for clave, cfg in RANKINGS.items():
        print(f'\n=== {cfg["nombre"]} ===')
        resultados[clave] = scrapear_ranking(driver, cfg['url'])
        print(f'  · Filas recolectadas: {len(resultados[clave])}')
finally:
    driver.quit()

# Vista rápida
for k, df in resultados.items():
    print(f'\n>>> {k} ({len(df)} filas)')
    display(df.head())

### 4.6 (Opcional) Enriquecer con la dirección estructurada

Cada universidad tiene una página de perfil que incluye JSON-LD con su dirección. Extraemos los componentes estructurados (`street_address`, `locality`, `region`, `postal_code`, `country_address`). Esta parte es lenta porque hace una petición por universidad; puedes desactivarla con `OBTENER_DIRECCIONES = False`.

> _Nota_: se eliminó el campo `full_address` (dirección completa concatenada) porque ya no aporta información adicional respecto a los componentes estructurados.

In [ ]:
OBTENER_DIRECCIONES = True  # <- cambia a False para omitir esta sección

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36'
}


def _buscar_en_json(obj, clave):
    """Reemplazo minimalista de `objectpath`: busca una clave a cualquier profundidad."""
    if isinstance(obj, dict):
        if clave in obj:
            return obj[clave]
        for v in obj.values():
            r = _buscar_en_json(v, clave)
            if r is not None:
                return r
    elif isinstance(obj, list):
        for v in obj:
            r = _buscar_en_json(v, clave)
            if r is not None:
                return r
    return None


def obtener_direccion(url_perfil: str, timeout: int = 15) -> dict:
    vacio = {'street_address': None, 'locality': None, 'region': None,
             'postal_code': None, 'country_address': None}
    if not url_perfil:
        return vacio
    try:
        r = requests.get(url_perfil, headers=HEADERS, timeout=timeout)
        r.raise_for_status()
        pagina = soup(r.text, 'html.parser')
        jsonld = None
        for tag in pagina.find_all('script', {'type': 'application/ld+json'}):
            try:
                jsonld = json.loads(tag.string or '{}')
                if _buscar_en_json(jsonld, 'address') is not None:
                    break
            except Exception:
                jsonld = None
        direccion = _buscar_en_json(jsonld or {}, 'address') or {}
        if not isinstance(direccion, dict):
            direccion = {}
        return {
            'street_address':  direccion.get('streetAddress'),
            'locality':        direccion.get('addressLocality'),
            'region':          direccion.get('addressRegion'),
            'postal_code':     direccion.get('postalCode'),
            'country_address': direccion.get('addressCountry'),
        }
    except Exception as e:
        print(f'  ! {url_perfil} -> {e}')
        return vacio


if OBTENER_DIRECCIONES:
    for clave, df in resultados.items():
        if df.empty or 'url' not in df.columns:
            continue
        print(f'\n>>> Enriqueciendo direcciones para {clave} ({len(df)} universidades)')
        direcciones = []
        for i, u in enumerate(df['url'].tolist(), 1):
            direcciones.append(obtener_direccion(u))
            if i % 25 == 0:
                print(f'   {i}/{len(df)} listos')
        df_addr = pd.DataFrame(direcciones)
        resultados[clave] = pd.concat([df.reset_index(drop=True), df_addr.reset_index(drop=True)], axis=1)

### 4.7 Limpieza y normalización

Conservamos el `rank` original como `rank_raw` y generamos un `rank` numérico limpio (sin `=`, `+`, ni rangos tipo `201–250`).

In [ ]:
COLUMNAS_FINALES = [
    'rank_raw', 'rank', 'name', 'country',
    'overall_score', 'teaching_score', 'research_env_score',
    'research_quality_score', 'industry_score', 'international_score',
    'url',
    'street_address', 'locality', 'region', 'postal_code', 'country_address',
]


def limpiar(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df = df.copy()
    df['rank_raw'] = df['rank']
    df['rank'] = (df['rank'].astype(str)
                           .str.replace(r'[=+]', '', regex=True)
                           .str.replace(r'[–-]\d+', '', regex=True)
                           .str.strip())
    if 'overall_score' in df.columns:
        df['overall_score'] = df['overall_score'].astype(str).str.replace(r'.*[–-]', '', regex=True)
    df = df.replace({'n/a': None, 'N/A': None, '': None, 'None': None})
    # Reordenamos sólo con las columnas que realmente existan
    presentes = [c for c in COLUMNAS_FINALES if c in df.columns]
    return df[presentes]


resultados = {k: limpiar(v) for k, v in resultados.items()}
for k, df in resultados.items():
    print(f'{k}: {df.shape}')
    display(df.head())

### 4.8 Guardar los resultados

Un `.csv` por ranking, en UTF-8 con BOM para que Excel lo abra sin problemas de codificación.

In [ ]:
for clave, cfg in RANKINGS.items():
    df = resultados.get(clave)
    if df is None or df.empty:
        print(f'! {clave} sin datos, se omite')
        continue
    salida = cfg['archivo_salida']
    df.to_csv(salida, index=False, encoding='utf-8-sig')
    print(f'✔ {clave:25s} -> {salida} ({len(df)} filas)')

## 5. Próximos pasos

* Comparar los pilares (`teaching`, `research_env`, `research_quality`, `industry`, `international`) entre los tres rankings.
* Cruzar con datos internos de la Universidad del Pacífico para análisis competitivo.
* Agendar una ejecución periódica (p. ej. mensual) para detectar cambios.

Si THE vuelve a cambiar la estructura del sitio, el único punto a ajustar es la función `parsear_filas`: como trabaja por **posición de columna** (no por clases CSS), resiste bien los cambios cosméticos del markup.